<a href="https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row =One row represents one content page for one client on one report date. The working slice is March 2026, restricted to rows where GSC data is available.*

In [3]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')"
)

print("Hugging Face secret configured.")

Hugging Face secret configured.


In [4]:
con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [5]:
con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Fields: feature / label / context / excluded

**gsc_impressions** — knowable at the decision moment because it is already observed in the selected historical window.

**gsc_clicks** — knowable at the decision moment because clicks are already recorded before the recommendation is made.

**gsc_avg_position** — knowable at the decision moment because it summarizes observed search visibility in the historical window.

**ga4_sessions** — knowable at the decision moment when GA4 is available for that page/client.

**ga4_engaged_sessions** — knowable at the decision moment when GA4 is available; rows without GA4 availability should not be treated as zero engagement.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The grain is one client-page-date observation. In March 2026, the total row count and unique client-page-date key count are both 3,611,061, so there are no duplicate keys in this slice.

In [8]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
        AS unique_client_page_date_keys
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────────────┐
│ total_rows │ unique_client_page_date_keys │
│   int64    │            int64             │
├────────────┼──────────────────────────────┤
│    3611061 │                      3611061 │
└────────────┴──────────────────────────────┘

The March 2026 slice contains 3,611,061 observations covering March 1 through March 31, across 47 clients and 176,738 pages.

In [6]:
month = "2026-03"

fact_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

query = f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS pages
FROM read_parquet('{fact_path}', hive_partitioning=1)
WHERE month = '{month}'
  AND gsc_data_available IS TRUE
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┬─────────┬────────┐
│  rows   │ first_date │ last_date  │ clients │ pages  │
│  int64  │    date    │    date    │  int64  │ int64  │
├─────────┼────────────┼────────────┼─────────┼────────┤
│ 3611061 │ 2026-03-01 │ 2026-03-31 │      47 │ 176738 │
└─────────┴────────────┴────────────┴─────────┴────────┘

GSC availability is TRUE for all 3,611,061 March rows, while GA4 availability is TRUE for 413,966 rows. Requiring both sources leaves 364,347 observations. This means my main search-performance slice can use GSC fields, while GA4-based features would substantially reduce the available data.

In [7]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_gsc_filter,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS rows_surviving_both_filter
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┬───────────────────────────┬────────────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ rows_surviving_gsc_filter │ rows_surviving_both_filter │
│   int64    │       int128       │       int128       │           int64           │           int64            │
├────────────┼────────────────────┼────────────────────┼───────────────────────────┼────────────────────────────┤
│    9841378 │            3611061 │             413966 │                   3611061 │                     364347 │
└────────────┴────────────────────┴────────────────────┴───────────────────────────┴────────────────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data limits**

This dataset can describe observed search and analytics performance, but it cannot establish that a refresh caused a page to improve or decline. The March slice is also only one month of daily observations, so it does not by itself capture a full before-and-after outcome window. GSC and GA4 availability are uneven, so analyses requiring GA4 will use fewer rows. The data also contains overlapping daily observations for the same page across time, so I need to avoid treating those rows as independent pages when evaluating a model. Finally, a high-priority score would indicate a page worth review, not guarantee that a refresh will produce better performance.

In [10]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT content_hash_id) AS pages,
    COUNT(DISTINCT report_date) AS dates,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────┬───────┬───────────────┬───────────────┐
│  rows   │ pages  │ dates │ gsc_available │ ga4_available │
│  int64  │ int64  │ int64 │     int64     │     int64     │
├─────────┼────────┼───────┼───────────────┼───────────────┤
│ 9841378 │ 331437 │    31 │       3611061 │        413966 │
└─────────┴────────┴───────┴───────────────┴───────────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.